In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "stccobankanticsdevcaeast";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
df = spark.read.csv(f"abfss://exlt-golden@{storageName}.dfs.core.windows.net/golden/divipola.csv")
df.display()

_c0,_c1,_c2,_c3,_c4,_c5,_c6
Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"
05,ANTIOQUIA,05031,AMALFI,Municipio,"-75,077501","6,909655"
05,ANTIOQUIA,05034,ANDES,Municipio,"-75,878828","5,657194"
05,ANTIOQUIA,05036,ANGELÓPOLIS,Municipio,"-75,711389","6,109719"
05,ANTIOQUIA,05038,ANGOSTURA,Municipio,"-75,335116","6,885175"


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://mts-co-bank-analytics-dev-canadaeast@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://exlt-raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://exlt-bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://exlt-silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://exlt-golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
--MANAGED LOCATION 'abfss://exlt-metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://exlt-bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://exlt-silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://exlt-golden@{storageName}.dfs.core.windows.net/",True)

True

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE TABLE IF NOT EXISTS catalog_au.bronze.sfc_indicadores_raw
(
    archivo_origen      STRING,
    anio_origen         INT,
    hoja                STRING,
    fila_excel          INT,
    columna_excel       INT,
    coordenada_excel    STRING,
    valor_texto         STRING,
    tipo_valor          STRING,
    fecha_ingesta       TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-bronze@${storageName}.dfs.core.windows.net/indicadores"
COMMENT 'Datos originales de indicadores financieros SFC, conservados a nivel de celda Excel para auditoria y reprocesamiento';

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;

CREATE TABLE IF NOT EXISTS catalog_au.bronze.banrep_indicadores_raw
(
    id_registro         STRING,
    codigo_serie        STRING,
    nombre_serie        STRING,
    fecha_observacion   STRING,
    valor_original      STRING,
    unidad_original     STRING,
    fuente_archivo      STRING,
    fecha_ingesta       TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-bronze@${storageName}.dfs.core.windows.net/banrep_indicadores"
COMMENT 'Datos economicos originales del Banco de la Republica de Colombia';

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;

CREATE TABLE IF NOT EXISTS catalog_au.bronze.sfc_archivos_control
(
    id_archivo          STRING,
    archivo_origen      STRING,
    ruta_archivo        STRING,
    anio_origen         INT,
    fecha_archivo       DATE,
    tamanio_bytes       BIGINT,
    total_filas         BIGINT,
    total_columnas      INT,
    total_celdas        BIGINT,
    estado_proceso      STRING,
    fecha_inicio        TIMESTAMP,
    fecha_fin           TIMESTAMP,
    mensaje_error       STRING
)
USING DELTA
LOCATION "abfss://exlt-bronze@${storageName}.dfs.core.windows.net/control_archivos"
COMMENT 'Control y trazabilidad de archivos Excel de la Superintendencia Financiera de Colombia';

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;

CREATE TABLE IF NOT EXISTS catalog_au.bronze.entidades_financieras_raw
(
    id_entidad_origen   STRING,
    codigo_entidad      STRING,
    nit                 STRING,
    nombre_entidad      STRING,
    tipo_entidad        STRING,
    estado_entidad      STRING,
    fecha_registro      STRING,
    sistema_origen      STRING,
    fecha_ingesta       TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-bronze@${storageName}.dfs.core.windows.net/entidades_financieras"
COMMENT 'Catalogo original de entidades financieras procedente de sistemas fuente';

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;

CREATE TABLE IF NOT EXISTS catalog_au.bronze.bronze_ingestion_log
(
    id_ejecucion          STRING,
    nombre_pipeline       STRING,
    tabla_destino         STRING,
    fuente_datos          STRING,
    fecha_inicio          TIMESTAMP,
    fecha_fin             TIMESTAMP,
    registros_leidos      BIGINT,
    registros_insertados  BIGINT,
    registros_rechazados  BIGINT,
    estado_ejecucion      STRING,
    mensaje_error         STRING
)
USING DELTA
LOCATION "abfss://exlt-bronze@${storageName}.dfs.core.windows.net/ingestion_log"
COMMENT 'Registro de auditoria y monitoreo de ejecuciones de ingesta de datos en la capa Bronze';

In [0]:
%sql

SHOW TABLES IN catalog_au.bronze;

database,tableName,isTemporary
bronze,banrep_indicadores_raw,false
bronze,bronze_ingestion_log,false
bronze,entidades_financieras_raw,false
bronze,sfc_archivos_control,false
bronze,sfc_indicadores_raw,false
,_sqldf,true


In [0]:
%sql

DESCRIBE DETAIL catalog_au.bronze.sfc_indicadores_raw;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ebd1e191-f685-4d14-a58b-35a1aa0f1c9e,catalog_au.bronze.sfc_indicadores_raw,"Datos originales de indicadores financieros SFC, conservados a nivel de celda Excel para auditoria y reprocesamiento",abfss://exlt-bronze@stccobankanticsdevcaeast.dfs.core.windows.net/indicadores,2026-11-11T20:17:39.521Z,2026-11-11T20:17:40Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


###Tablas Silver

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.silver;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS catalog_au.silver.sfc_indicadores
(
    id_indicador           STRING,
    codigo_entidad         STRING,
    nombre_entidad         STRING,
    codigo_concepto        STRING,
    nombre_concepto        STRING,
    fecha_corte            DATE,
    anio                   INT,
    mes                    INT,
    valor_indicador        DECIMAL(28,6),
    unidad_medida          STRING,
    archivo_origen         STRING,
    fecha_ingesta          TIMESTAMP,
    fecha_transformacion   TIMESTAMP,
    estado_validacion      STRING
)
USING DELTA
LOCATION "abfss://exlt-silver@${storageName}.dfs.core.windows.net/sfc_indicadores"
COMMENT 'Indicadores financieros SFC normalizados, tipificados y validados para analisis bancario';

In [0]:
%sql

CREATE TABLE IF NOT EXISTS catalog_au.silver.banrep_indicadores
(
    id_indicador           STRING,
    codigo_serie          STRING,
    nombre_serie          STRING,
    fecha_observacion     DATE,
    anio                  INT,
    mes                   INT,
    valor_indicador       DECIMAL(28,6),
    unidad_medida         STRING,
    frecuencia            STRING,
    fuente_origen         STRING,
    fecha_ingesta         TIMESTAMP,
    fecha_transformacion  TIMESTAMP,
    estado_validacion     STRING
)
USING DELTA
LOCATION "abfss://exlt-silver@${storageName}.dfs.core.windows.net/banrep_indicadores"
COMMENT 'Series economicas del Banco de la Republica normalizadas para analisis financiero';

In [0]:
%sql

CREATE TABLE IF NOT EXISTS catalog_au.silver.entidades_financieras
(
    id_entidad             STRING,
    codigo_entidad         STRING,
    nit                    STRING,
    nombre_entidad         STRING,
    nombre_normalizado     STRING,
    tipo_entidad           STRING,
    estado_entidad         STRING,
    fecha_inicio_vigencia  DATE,
    fecha_fin_vigencia     DATE,
    sistema_origen         STRING,
    fecha_ingesta          TIMESTAMP,
    fecha_transformacion   TIMESTAMP,
    estado_validacion      STRING
)
USING DELTA
LOCATION "abfss://exlt-silver@${storageName}.dfs.core.windows.net/entidades_financieras"
COMMENT 'Catalogo estandarizado de entidades financieras colombianas con fechas de vigencia';

In [0]:
%sql

CREATE TABLE IF NOT EXISTS catalog_au.silver.silver_calidad_datos
(
    id_validacion          STRING,
    nombre_pipeline       STRING,
    tabla_origen          STRING,
    tabla_destino         STRING,
    regla_validacion      STRING,
    tipo_validacion       STRING,
    registros_evaluados   BIGINT,
    registros_validos     BIGINT,
    registros_invalidos   BIGINT,
    porcentaje_calidad    DECIMAL(5,2),
    estado_validacion     STRING,
    descripcion_error     STRING,
    fecha_validacion      TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-silver@${storageName}.dfs.core.windows.net/calidad_datos"
COMMENT 'Auditoria y resultados de controles de calidad aplicados durante la transformacion Silver';

In [0]:
%sql

SHOW TABLES IN catalog_au.silver;

database,tableName,isTemporary
silver,banrep_indicadores,false
silver,entidades_financieras,false
silver,sfc_indicadores,false
silver,silver_calidad_datos,false
,_sqldf,true


In [0]:
%sql

SELECT
    table_name,
    table_type,
    data_source_format,
    storage_path
FROM catalog_au.information_schema.tables
WHERE table_schema = 'silver'
ORDER BY table_name;

table_name,table_type,data_source_format,storage_path
banrep_indicadores,EXTERNAL,DELTA,abfss://exlt-silver@stccobankanticsdevcaeast.dfs.core.windows.net/banrep_indicadores
entidades_financieras,EXTERNAL,DELTA,abfss://exlt-silver@stccobankanticsdevcaeast.dfs.core.windows.net/entidades_financieras
sfc_indicadores,EXTERNAL,DELTA,abfss://exlt-silver@stccobankanticsdevcaeast.dfs.core.windows.net/sfc_indicadores
silver_calidad_datos,EXTERNAL,DELTA,abfss://exlt-silver@stccobankanticsdevcaeast.dfs.core.windows.net/calidad_datos


###Tablas Golden

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.gold;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS catalog_au.golden.dim_entidad
(
    sk_entidad           BIGINT GENERATED ALWAYS AS IDENTITY,
    id_entidad           STRING,
    codigo_entidad       STRING,
    nit                  STRING,
    nombre_entidad       STRING,
    nombre_normalizado   STRING,
    tipo_entidad         STRING,
    estado_entidad       STRING,
    fecha_inicio_vigencia DATE,
    fecha_fin_vigencia   DATE,
    es_actual            BOOLEAN,
    fecha_creacion       TIMESTAMP,
    fecha_actualizacion  TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-golden@${storageName}.dfs.core.windows.net/dim_entidad"
COMMENT 'Dimension historica de entidades financieras colombianas con gestion SCD Tipo 2';

In [0]:
%sql

DESCRIBE DETAIL catalog_au.golden.dim_entidad;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,d82a5763-3553-47bf-acc1-87f432bf854d,catalog_au.golden.dim_entidad,Dimension historica de entidades financieras colombianas con gestion SCD Tipo 2,abfss://exlt-golden@stccobankanticsdevcaeast.dfs.core.windows.net/dim_entidad,2026-11-11T20:43:09.696Z,2026-11-11T20:43:09Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, identityColumns, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

CREATE TABLE IF NOT EXISTS catalog_au.golden.dim_tiempo
(
    sk_tiempo           INT,
    fecha               DATE,
    anio                INT,
    semestre            INT,
    trimestre           INT,
    mes                 INT,
    nombre_mes          STRING,
    dia                 INT,
    dia_semana          INT,
    nombre_dia          STRING,
    es_fin_mes          BOOLEAN,
    es_fin_trimestre    BOOLEAN,
    es_fin_anio         BOOLEAN,
    fecha_creacion      TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-golden@${storageName}.dfs.core.windows.net/dim_tiempo"
COMMENT 'Dimension calendario para analisis de indicadores financieros por fecha, mes, trimestre y anio';

In [0]:
%sql

DESCRIBE DETAIL catalog_au.golden.dim_tiempo;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,591c5877-9802-4b1e-860f-fc247507bf20,catalog_au.golden.dim_tiempo,"Dimension calendario para analisis de indicadores financieros por fecha, mes, trimestre y anio",abfss://exlt-golden@stccobankanticsdevcaeast.dfs.core.windows.net/dim_tiempo,2026-11-11T20:44:39.792Z,2026-11-11T20:44:39Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

CREATE TABLE IF NOT EXISTS catalog_au.golden.dim_concepto_financiero
(
    sk_concepto           BIGINT GENERATED ALWAYS AS IDENTITY,
    codigo_concepto       STRING,
    nombre_concepto       STRING,
    codigo_padre          STRING,
    nivel_jerarquia       INT,
    ruta_jerarquia        STRING,
    categoria_financiera  STRING,
    unidad_medida         STRING,
    es_concepto_hoja      BOOLEAN,
    estado_concepto       STRING,
    fecha_creacion        TIMESTAMP,
    fecha_actualizacion   TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-golden@${storageName}.dfs.core.windows.net/dim_concepto_financiero"
COMMENT 'Dimension jerarquica de conceptos financieros de la Superintendencia Financiera de Colombia';

In [0]:
%sql

DESCRIBE DETAIL catalog_au.golden.dim_concepto_financiero;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,db82efd3-9a9b-460d-96c6-0038f3dbe049,catalog_au.golden.dim_concepto_financiero,Dimension jerarquica de conceptos financieros de la Superintendencia Financiera de Colombia,abfss://exlt-golden@stccobankanticsdevcaeast.dfs.core.windows.net/dim_concepto_financiero,2026-11-11T20:46:23.926Z,2026-11-11T20:46:24Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, identityColumns, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

CREATE TABLE IF NOT EXISTS catalog_au.golden.fact_indicadores_financieros
(
    id_hecho               STRING,
    sk_entidad             BIGINT,
    sk_concepto            BIGINT,
    sk_tiempo              INT,
    fecha_corte            DATE,
    valor_indicador        DECIMAL(28,6),
    unidad_medida          STRING,
    archivo_origen         STRING,
    fecha_ingesta          TIMESTAMP,
    fecha_procesamiento    TIMESTAMP
)
USING DELTA
LOCATION "abfss://exlt-golden@${storageName}.dfs.core.windows.net/fact_indicadores_financieros"
COMMENT 'Tabla de hechos de indicadores financieros SFC relacionados con las dimensiones entidad, concepto y tiempo';

In [0]:
%sql

DESCRIBE DETAIL catalog_au.golden.fact_indicadores_financieros;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cf2cf450-f81d-499d-9cc2-ee28b569dc51,catalog_au.golden.fact_indicadores_financieros,"Tabla de hechos de indicadores financieros SFC relacionados con las dimensiones entidad, concepto y tiempo",abfss://exlt-golden@stccobankanticsdevcaeast.dfs.core.windows.net/fact_indicadores_financieros,2026-11-11T20:47:41.329Z,2026-11-11T20:47:41Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

SHOW TABLES IN catalog_au.golden;

database,tableName,isTemporary
golden,dim_concepto_financiero,false
golden,dim_entidad,false
golden,dim_tiempo,false
golden,fact_indicadores_financieros,false
,_sqldf,true
